> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업의 실습이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🚀 인공지능 프로젝트 틀

| 단계 | 할 일 | 사이트 |
|---|---|---|
| 0 | 문제 정하기 | 🤖 개념 |
| 1 | 데이터 모으기 | 🕸 수집 |
| 2 | 살펴보기(결측치·이상치) | 🔍 가공 |
| 3 | 전처리 | 🧹 전처리 |
| 4 | 나누기 | 🧩 학습 준비 |
| 5 | 모델 학습 | 🤖 기계학습 |
| 6 | 평가 | 🚀 모델 평가 |
| 7 | 결론 | — |

이 노트북은 **펭귄의 종 맞히기** 예시로 채워져 있어요. 모든 셀을 실행하면 끝까지 돌아가요.
"✏️ 여기를 바꿔요" 셀만 내 주제에 맞게 고쳐 쓰세요.

사이트: https://yanghyeokjin33.github.io/Algorithm/

> **Colab 사용법** — 셀 왼쪽의 ▶ 단추(또는 **Shift + Enter**)를 누르면 그 셀이 실행돼요.
> 위에서부터 **차례로** 실행하세요. 고쳐 보고 싶으면 먼저 **파일 → 드라이브에 사본 저장**을 눌러 내 것으로 만드세요.
> 파이썬을 몰라도 괜찮아요. 코드 위의 설명과 사이트의 의사코드를 짝지어 읽으면 돼요.

## 0. 문제 정하기 ✏️

- **무엇을 맞힐까?** 펭귄의 종 (아델리·턱끈·젠투)
- **학습 목적**: 분류 (예측이면 숫자, 군집이면 정답 없음)
- **정답(y)**: '종' 열 · **입력(X)**: 부리길이, 부리깊이, 날개길이, 몸무게
- **데이터 출처**: palmerpenguins (CC0) — 수업용 연습 사이트

## 1. 데이터 모으기 ✏️

- 내 데이터를 쓸 때는 DATA_URL, TARGET, FEATURES 세 줄만 바꾸면 돼요.
- 크롤링이 필요하면 01번 노트북의 crawl_page 함수를 가져와 쓰세요.

In [ ]:
import pandas as pd
DATA_URL = 'https://raw.githubusercontent.com/YANGHYEOKJIN33/Algorithm/main/data/penguins.csv'      # ✏️ 여기를 바꿔요: 내 CSV 주소나 파일 이름
df = pd.read_csv(DATA_URL)
TARGET = '종'                                              # ✏️ 정답 열
FEATURES = ['부리길이', '부리깊이', '날개길이', '몸무게']  # ✏️ 입력 열
print(df.shape)
df.head()

## 2. 데이터 살펴보기 — 결측치·이상치

- 빈칸·겹친 행·이상한 최솟값/최댓값이 있는지 확인해요.

In [ ]:
print('결측치:'); print(df[FEATURES + [TARGET]].isnull().sum())
print('겹친 행:', df.duplicated().sum())
df[FEATURES].describe()

- 그래프에 한글(제목·축 이름)이 깨지지 않도록 글꼴 도구를 설치해요. 처음 한 번 1~2분 걸릴 수 있어요.

In [ ]:
!pip install -q koreanize-matplotlib
try:
    import koreanize_matplotlib  # 그래프에 한글 글꼴을 입혀 줘요
except ImportError:
    print('한글 글꼴을 불러오지 못했어요. 그래프 글자가 네모로 보일 수 있어요.')
import matplotlib.pyplot as plt

- 속성마다 상자그림을 그려 이상치를 찾아요.

In [ ]:
df[FEATURES].plot.box(subplots=True, layout=(1, len(FEATURES)), figsize=(12, 3))
plt.tight_layout(); plt.show()

## 3. 전처리 ✏️

- between(low, high)는 울타리 안에 있는지 확인해요.
- 정답(y)이 비어 있으면 배울 수 없으니 지워요.
- 글자로 된 입력 속성이 있다면 map으로 숫자로 바꾸세요(03번 노트북).

In [ ]:
df = df.drop_duplicates()                                   # 겹친 행 지우기
for col in FEATURES:                                        # 이상치(울타리 밖) 지우기
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    df = df[df[col].isnull() | df[col].between(low, high)]
df = df.dropna(subset=FEATURES, how='all')                  # 측정값이 모두 빈 행 지우기
df[FEATURES] = df[FEATURES].fillna(df[FEATURES].mean())     # 숫자 빈칸 → 평균
df = df.dropna(subset=[TARGET])                             # 정답이 빈 행은 지우기
print(df.shape, '/ 남은 빈칸:', df[FEATURES + [TARGET]].isnull().sum().sum())

## 4. 나누기

- 테스트 데이터는 마지막 평가 때까지 쓰지 않아요.

In [ ]:
from sklearn.model_selection import train_test_split
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('훈련', X_train.shape, '/ 테스트', X_test.shape)

## 5. 모델 학습 ✏️ — 목적에 맞게 고르기

| 목적 | 알고리즘 | 만들기 |
|---|---|---|
| 분류 | k-최근접 이웃 | `KNeighborsClassifier(n_neighbors=5)` |
| 분류 | 의사결정 트리 | `DecisionTreeClassifier(max_depth=3)` |
| 예측 | 선형 회귀 | `LinearRegression()` |
| 군집 | k-평균 | `KMeans(n_clusters=3, n_init=10)` (y 없이 fit(X)) |

- 두 가지 이상 견줘 보면 발표가 더 탄탄해져요.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

models = {                                      # ✏️ 견줄 모델들
    'k-최근접 이웃 (k=5)': KNeighborsClassifier(n_neighbors=5),
    '의사결정 트리 (깊이 3)': DecisionTreeClassifier(max_depth=3, random_state=0),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    print(name, '학습 완료')

## 6. 평가

- 정확도 = 맞힌 개수 ÷ 테스트 개수 (예측 문제라면 mean_squared_error를 써요).

In [ ]:
from sklearn.metrics import accuracy_score
for name, m in models.items():
    pred = m.predict(X_test)
    print(f'{name}: 정확도 {accuracy_score(y_test, pred):.3f}')

- 틀린 행을 모아 보면 어떤 경우에 헷갈리는지 보여요. 결론에 쓰기 좋아요.

In [ ]:
# 틀린 예 살펴보기 — 모델의 약점 찾기
best = models['의사결정 트리 (깊이 3)']
result = X_test.copy()
result['정답'] = y_test
result['예측'] = best.predict(X_test)
result[result['정답'] != result['예측']]

## 7. 결론 ✏️

- **결과**: 어떤 모델이 정확도 몇 %였나?
- **까닭**: 왜 그 모델이 더 잘 맞았을까? 어떤 속성이 중요했나?
- **한계**: 데이터가 적거나 치우친 점, 틀린 예의 공통점
- **개선**: 속성을 더하거나, 데이터를 더 모으거나, 다른 알고리즘을 써 본다면?